# Study 817 — Realized-Volatility Trend 📈📉

**Does *rising* volatility keep de-rating a stock — and *falling* volatility re-rate it?**

Two names can share the same volatility *level* yet be moving in opposite directions:
one's vol is climbing, the other's is cooling. This study sorts on that **trend** —
each name's `(trailing 21d realized vol) / (trailing 63d realized vol) - 1` — long the
**falling-vol** names, short the **rising-vol** ones, and asks the honest question:
is this vol *momentum* anything beyond the low-vol *level* anomaly (study 330)? We take
the daily version on a liquid US cross-section (2010-01-04 → 2026-06-30,
50 names).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — magnitudes are an upper
bound.*


## 1. The idea in one picture

When a name's *short-window* realized vol pulls **above** its longer-window average, its risk is being re-priced upward — and, the story goes, the equity de-rates with it. When vol is **cooling** the opposite: the risk premium relaxes and the name re-rates. So buy the falling-vol names, sell the rising-vol ones. Note this is the *change* in vol, deliberately built as a ratio so it is near-orthogonal to the vol **level** (the low-vol anomaly).

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=0.94, t_nw=0.86, lo_bps=7.74, hi_bps=6.8, gross_sharpe=0.21,
         corr=0.065, alpha_bps=1.12, alpha_t=1.02)
print('long falling-vol / short rising-vol spread: %+.2f bps/day (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('  falling-vol book %+.2f bps vs rising-vol book %+.2f bps'
      % (R['lo_bps'], R['hi_bps']))
print('  gross spread Sharpe (before cost): %.2f' % R['gross_sharpe'])
print('  corr with low-vol LEVEL sort: %+.3f  (near-orthogonal)' % R['corr'])
print('  trend alpha net of the level anomaly: %+.2f bps/day (NW t = %+.2f)'
      % (R['alpha_bps'], R['alpha_t']))

long falling-vol / short rising-vol spread: +0.94 bps/day (NW t = +0.86)
  falling-vol book +7.74 bps vs rising-vol book +6.80 bps
  gross spread Sharpe (before cost): 0.21
  corr with low-vol LEVEL sort: +0.065  (near-orthogonal)
  trend alpha net of the level anomaly: +1.12 bps/day (NW t = +1.02)


## 2. Is the sort just lucky? A live synthetic control

We plant the effect in a seeded toy world (`edge>0`) and check the detector recovers it — and that it stays *silent* on the null (`edge=0`, vol trend present but unpriced). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from vol_trend import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=817, n_assets=40, n_days=1200))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0015, seed=817, n_assets=40, n_days=1500))
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up)' % planted['t_nw'])

null world   : spread NW t = +1.40  (should be ~0)
planted world: spread NW t = +9.49  (should light up)


## 3. The honest verdict — the trend says nothing here

On this liquid mega-cap tape the long-falling-vol / short-rising-vol spread is **+0.94 bps/day** with NW *t* = **+0.86** — the *claimed* sign, but statistically indistinguishable from zero (the permutation null centres at 0 with sd 0.88 bps; the observed value is only ~1.1σ into the right tail, p = 0.12). And it is **not additive**: near-orthogonal to the low-vol *level* sort (corr +0.065) yet its alpha net of that level is just **+1.12 bps/day** (*t* = +1.02). The seeded synthetic control recovers a *planted* trend relation cleanly, so this is a genuine absence of edge, not a bug. **Signal: None**, **Tradability: Mirage**.